In [12]:
import numpy as np
import random

def get_reward(action, true_goal):
    g1_effort, g2_effort = action

    if true_goal == 'g1':
        if g1_effort > g2_effort:
            return g1_effort  # Reward is what is consumed at g1 when it's more than g2
        elif g1_effort < g2_effort:
            return -1  # Penalty if more is consumed at g2 when g1 is the true goal
        else:
            return 0  # Return 0 if efforts are equally split
    else:  # true_goal is 'g2'
        if g2_effort > g1_effort:
            return g2_effort  # Reward is what is consumed at g2 when it's more than g1
        elif g2_effort < g1_effort:
            return -1  # Penalty if more is consumed at g1 when g2 is the true goal
        else:
            return 0  # Return 0 if efforts are equally split




In [13]:
movements = {
    'up': (-1, 0),
    'down': (1, 0),
    'left': (0, -1),
    'right': (0, 1),
    'none': (0, 0),  # No movement

}


In [14]:
def update_position(position, move):
    dx, dy = movements.get(move, (0, 0))
    return (position[0] + dx, position[1] + dy)

In [15]:
def simulate_step(position, bananas_at_g1, bananas_at_g2, goal_positions, true_goal, step_count):
    mover_action = mover_policy(position, goal_positions, step_count)  # Assuming this function is defined
    new_position = update_position(position, mover_action)
    eater_action = select_action(new_position, goal_positions, bananas_at_g1, bananas_at_g2)
    new_bananas_at_g1, new_bananas_at_g2 = environment(eater_action, bananas_at_g1, bananas_at_g2)
    reward = get_reward(eater_action, true_goal)
    return new_position, new_bananas_at_g1, new_bananas_at_g2, reward, mover_action, eater_action


In [26]:
def environment(eater_action, bananas_at_g1, bananas_at_g2):
    # Calculate how many bananas are consumed at G1 and G2 based on the eater's action
    consumed_g1 = eater_action[0]
    consumed_g2 = eater_action[1]

    # Update the banana counts after consumption
    bananas_at_g1 = max(0, bananas_at_g1 - consumed_g1)
    bananas_at_g2 = max(0, bananas_at_g2 - consumed_g2)

    #print(f"Consuming {consumed_g1} bananas at G1. Remaining: {bananas_at_g1}")
    #print(f"Consuming {consumed_g2} bananas at G2. Remaining: {bananas_at_g2}")

    return bananas_at_g1, bananas_at_g2


In [22]:
def mover_policy(position, goal_positions, step_count):
    # Determine the current goal based on some logic, e.g., cycle through goals or based on proximity
    current_goal = goal_positions[step_count % len(goal_positions)]  # This cycles through the goals
    if position[0] < current_goal[0]:
        return 'down'
    elif position[0] > current_goal[0]:
        return 'up'
    elif position[1] < current_goal[1]:
        return 'right'
    elif position[1] > current_goal[1]:
        return 'left'
    return 'none'  # If already at the goal


In [23]:
def select_action(position, goal_positions, bananas_at_g1, bananas_at_g2):
    g1_position, g2_position = goal_positions
    distance_to_g1 = np.linalg.norm(np.array(position) - np.array(g1_position))
    distance_to_g2 = np.linalg.norm(np.array(position) - np.array(g2_position))

    actions = [
        (1, 0),   # Consume all from g1
        (0, 1),   # Consume all from g2
        (0.5, 0.5),  # Equally consume from both
        (0.2, 0.8),  # Mainly consume from g2
        (0.6, 0.4),  # Mainly consume from g1
        (0.8, 0.2),  # Mainly consume from g1
        (0.4, 0.6),  # Mainly consume from g2
        (0.7, 0.3),  # Mainly consume from g1
        (0.3, 0.7)   # Mainly consume from g2
    ]

    probabilities = []
    for action in actions:
        g1_consumption = action[0] * bananas_at_g1
        g2_consumption = action[1] * bananas_at_g2
        distance_penalty = (distance_to_g1 * action[0] + distance_to_g2 * action[1]) / 2
        probability = (g1_consumption + g2_consumption) / (1 + distance_penalty)
        probabilities.append(probability)

    total_probability = sum(probabilities)
    if total_probability > 0:
        normalized_probabilities = [p / total_probability for p in probabilities]
    else:
        # Assign equal probability to each action when total probability is zero
        normalized_probabilities = [1 / len(actions)] * len(actions)

    action_index = np.random.choice(len(actions), p=normalized_probabilities)
    return actions[action_index]


In [27]:
# Initialization
position = (12, 2)
goal_positions = ((2, 1), (4, 5))
bananas_at_g1 = 20000
bananas_at_g2 = 20000
true_goals = 'g1' # Example of changing goals
step_count = 0
for _ in range(50):  # Simulate 5 steps
    position, bananas_at_g1, bananas_at_g2, reward, mover_action, eater_action = simulate_step(
        position, bananas_at_g1, bananas_at_g2, goal_positions, true_goals, step_count)
    print(f"Step {_+1}: Mover Action: {mover_action}, New Position: {position}, Eater Action: {eater_action}")
    print(f"Bananas at G1: {bananas_at_g1}, Bananas at G2: {bananas_at_g2}, Reward: {reward}")
    step_count += 1  # Increment step count for potential use in policies or other decisions

Step 1: Mover Action: up, New Position: (11, 2), Eater Action: (1, 0)
Bananas at G1: 19999, Bananas at G2: 20000, Reward: 1
Step 2: Mover Action: up, New Position: (10, 2), Eater Action: (0, 1)
Bananas at G1: 19999, Bananas at G2: 19999, Reward: -1
Step 3: Mover Action: up, New Position: (9, 2), Eater Action: (0.6, 0.4)
Bananas at G1: 19998.4, Bananas at G2: 19998.6, Reward: 0.6
Step 4: Mover Action: up, New Position: (8, 2), Eater Action: (0.3, 0.7)
Bananas at G1: 19998.100000000002, Bananas at G2: 19997.899999999998, Reward: -1
Step 5: Mover Action: up, New Position: (7, 2), Eater Action: (0.7, 0.3)
Bananas at G1: 19997.4, Bananas at G2: 19997.6, Reward: 0.7
Step 6: Mover Action: up, New Position: (6, 2), Eater Action: (0.5, 0.5)
Bananas at G1: 19996.9, Bananas at G2: 19997.1, Reward: 0
Step 7: Mover Action: up, New Position: (5, 2), Eater Action: (1, 0)
Bananas at G1: 19995.9, Bananas at G2: 19997.1, Reward: 1
Step 8: Mover Action: up, New Position: (4, 2), Eater Action: (0.7, 0.3)
